# TRÍCH XUẤT 600 THUỘC TÍNH HÌNH THÁI STANFORD DOGS TỪ GEMMA 4 (ON-DEVICE)
## Giao thức Chưng cất Tri thức Ngôn ngữ theo 5 Trục Giải Phẫu Học
### Đề tài: GEMMA-CLIP: Fine-Grained Pet Breed Classification with Morphological Knowledge, Prompt Optimization and Multi-View Inference
**Tác giả:** Nguyễn Hữu Hoàng Luân, Tôn Thất Vấn (Nhóm 1 - CV ST4 F106)

---
### Mục Tiêu:
* Tự động trích xuất các thuộc tính hình thái đặc trưng visible theo tiêu chuẩn AKC/FCI cho **120 giống chó** của bộ dữ liệu Stanford Dogs.
* Mỗi giống được mô tả trên **5 trục giải phẫu chuẩn hóa**:
  1. `head_and_ears`: Cấu trúc hộp sọ, mõm, tai.
  2. `eyes`: Màu sắc, hình dáng mắt.
  3. `coat`: Độ dài lông, chất lông, hoa văn màu sắc.
  4. `body`: Thể trạng, khung xương, đuôi, dáng đứng.
  5. `distinguishing_mark`: Dấu hiệu hình thái phân biệt đặc trưng nhất.
* Mô hình suy luận: `google/gemma-4-e4b` chạy cục bộ (on-device) qua LM Studio Local Server (nhiệt độ $T=0.0$).


In [1]:
import os
import sys
import json
import time
import urllib.request
import urllib.error

# Tự động chuyển về thư mục gốc của repository
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../..")) if "notebooks" in os.getcwd() else os.getcwd()
os.chdir(REPO_ROOT)
print(f"📁 Working Directory: {os.getcwd()}")

CLASSES_PATH = "Semester/cached_features/stanford_dogs/classes.json"
OUTPUT_FILE = "Semester/attributes/stanford_dogs_attributes.json"
os.makedirs("Semester/attributes", exist_ok=True)


📁 Working Directory: /Users/nguyenhuuhoangluan/Git_Code/CLIP-Animal-Classifier


In [2]:
def load_classes():
    with open(CLASSES_PATH, "r") as f:
        return json.load(f)

classes = load_classes()
print(f"✅ Đã nạp danh sách {len(classes)} giống chó Stanford Dogs.")
print(f"Ví dụ 5 giống đầu tiên: {classes[:5]}")


✅ Đã nạp danh sách 120 giống chó Stanford Dogs.
Ví dụ 5 giống đầu tiên: ['Affenpinscher', 'Afghan Hound', 'African Hunting Dog', 'Airedale', 'American Staffordshire Terrier']


In [3]:
def build_prompt(breed_name):
    return f"""You are a professional canine morphologist and veterinarian.
Directly output a valid JSON object describing the fine-grained visual characteristics of the dog breed: "{breed_name}".
Only provide visual characteristics visible in a clear photograph grounded in AKC/FCI standards.
Do NOT output reasoning. Output ONLY valid JSON:
{{
  "breed": "{breed_name}",
  "human_name": "{breed_name}",
  "species": "dog",
  "attributes": {{
    "head_and_ears": "specific shape of skull, muzzle, and ear carriage",
    "eyes": "distinctive eye color and eye shape",
    "coat": "coat texture, fur length, color patterns, or markings",
    "body": "overall body build, tail characteristics, stance",
    "distinguishing_mark": "unique visual trait separating it from similar dog breeds"
  }}
}}"""

print("Ví dụ Prompt gửi tới Gemma 4 SLM:")
print(build_prompt("Chihuahua"))


Ví dụ Prompt gửi tới Gemma 4 SLM:
You are a professional canine morphologist and veterinarian.
Directly output a valid JSON object describing the fine-grained visual characteristics of the dog breed: "Chihuahua".
Only provide visual characteristics visible in a clear photograph grounded in AKC/FCI standards.
Do NOT output reasoning. Output ONLY valid JSON:
{
  "breed": "Chihuahua",
  "human_name": "Chihuahua",
  "species": "dog",
  "attributes": {
    "head_and_ears": "specific shape of skull, muzzle, and ear carriage",
    "eyes": "distinctive eye color and eye shape",
    "coat": "coat texture, fur length, color patterns, or markings",
    "body": "overall body build, tail characteristics, stance",
    "distinguishing_mark": "unique visual trait separating it from similar dog breeds"
  }
}


In [4]:
def query_gemma_lm_studio(prompt, max_retries=3):
    url = "http://127.0.0.1:1234/v1/chat/completions"
    payload = {
        "model": "google/gemma-4-e4b",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0,
        "max_tokens": 1200,
        "stream": False
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=data,
        headers={"Content-Type": "application/json"}
    )
    for attempt in range(1, max_retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                res = json.loads(resp.read().decode("utf-8"))
                content = res["choices"][0]["message"].get("content", "").strip()
                if "```" in content:
                    parts = content.split("```")
                    for p in parts:
                        p = p.strip()
                        if p.startswith("json"):
                            p = p[4:].strip()
                        if p.startswith("{") and p.endswith("}"):
                            content = p
                            break
                elif "{" in content and "}" in content:
                    start = content.find("{")
                    end = content.rfind("}") + 1
                    content = content[start:end]
                parsed = json.loads(content)
                if "attributes" in parsed and all(k in parsed["attributes"] for k in ["head_and_ears", "eyes", "coat", "body", "distinguishing_mark"]):
                    return parsed
                else:
                    raise ValueError(f"Missing attributes in response: {parsed}")
        except Exception as e:
            if attempt == max_retries:
                raise e
            time.sleep(1.5)

print('✅ Đã sẵn sàng hàm query_gemma_lm_studio()')

✅ Đã sẵn sàng hàm query_gemma_lm_studio()


In [5]:
# Kiểm tra ngân hàng thuộc tính đã trích xuất hoàn chỉnh
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        attributes_bank = json.load(f)
    print(f"🎉 Đã nạp thành công {len(attributes_bank)}/{len(classes)} giống chó từ: {OUTPUT_FILE}")
    sample_breed = classes[0]
    print(f"\n🔍 Mẫu thuộc tính hình thái cho giống '{sample_breed}':")
    print(json.dumps(attributes_bank[sample_breed], indent=2, ensure_ascii=False))
else:
    print(f"⚠️ Chưa tìm thấy file {OUTPUT_FILE}, sẵn sàng chạy trích xuất qua LM Studio.")


🎉 Đã nạp thành công 120/120 giống chó từ: Semester/attributes/stanford_dogs_attributes.json

🔍 Mẫu thuộc tính hình thái cho giống 'Affenpinscher':
{
  "breed": "Affenpinscher",
  "human_name": "Affenpinscher",
  "species": "dog",
  "attributes": {
    "head_and_ears": "Strong, compressed skull; muzzle is proportionate and moderately short; ears are small, erect, and set high on the head.",
    "eyes": "Dark brown to black; almond-shaped, conveying intelligence and alertness.",
    "coat": "Dense double coat; texture is harsh yet manageable, lying close to the body; colors range from rich tan and reddish-brown to solid black or mixtures thereof.",
    "body": "Compact, sturdy, and muscular build; legs are strong and relatively short; overall gait is confident and balanced.",
    "distinguishing_mark": "The unique combination of the compressed, powerful skull structure and the alert, expressive facial carriage."
  }
}
